This notebook must be executed inside the FINN container provided by Xilinx.

Follow the steps of the [FINN docks](https://finn.readthedocs.io/en/latest/), section [Quickstart](https://finn.readthedocs.io/en/latest/getting_started.html#running-finn-in-docker), to download, build and verify the container installation.

You must also move your project folder to inside the same folder the repository is located.

To start the container, go to the folder where the repo was installed and run $ ./run-docker.sh notebook

If you are using vscode, you can select the notebook kernel inside the container to run the code.

This notebook is strongly based on Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook and 0BAB1 [2_finn_hardware_layers.ipynb](https://github.com/0BAB1/tutorial-snippets/blob/main/8%20Python%20to%20FPGA/2_finn_hardware_layers.ipynb) notebook.

Check them out for deeper instructions.

# Setup Python Paths for Libraries

In [1]:
import os
import sys

# Correct the path where the environment starts to be the same folder of the notebook, so that the imports work correctly.
# When starting the container with the notebook server, the script hardcodes the Jupyter server to start inside the ./notebooks folder.
os.chdir('../Optimized-QFast-SCNN-Lite_with_Brevitas_and_FINN/finn_environment')

# Add the train_environment directory to the system path to allow imports from there
sys.path.append(os.path.abspath('../train_environment'))
print(sys.path)

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/tmp/home_dir/.local/lib/python3.10/site-packages', '/home/jose-vitor/finn/deps/qonnx/src', '/home/jose-vitor/finn/deps/finn-experimental/src', '/home/jose-vitor/finn/deps/brevitas/src', '/home/jose-vitor/finn/src', '/usr/local/lib/python3.10/dist-packages', '/workspace/src/dataset-loading', '/usr/lib/python3/dist-packages', '/home/jose-vitor/finn/Optimized-QFast-SCNN-Lite_with_Brevitas_and_FINN/train_environment']


In [2]:
# Import all used libraries an functions and create all folders needed for the project, for convenience when is needed to run only a part of the project.
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.datasets import Cityscapes
from torch.utils.data import DataLoader
from pathlib import Path
import numpy as np
from config import IM_SIZE, NUM_CLASSES, DATA_PATH, BIT_WIDTH
from finn.util.visualization import showSrc, showInNetron
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.fold_constants import FoldConstants
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from onnx import helper
from my_finn_utils import load_state_dict, generate_cityscapes_labels, IdToTrainIdTransform
import models.QFastSCNN as qfscnn
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph
from finn.transformation.streamline import Streamline
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.reorder import MoveMulPastFork, MoveScalarMulPastConv, MoveScalarMulPastConvTranspose, MoveMulPastDWConv, MoveIdenticalOpPastJoinOp, MoveScalarLinearPastInvariants, MoveAddPastFork
from finn.transformation.streamline.absorb import AbsorbMulIntoMultiThreshold, AbsorbAddIntoMultiThreshold
from finn.util.basic import pynq_part_map
from qonnx.custom_op.registry import getCustomOp

# Setup Path
onnx_path = f'../onnx/quant_model_{BIT_WIDTH}_bits.onnx'
onnx_name = Path(onnx_path).stem

brevitas_onnx_path = f'./01_from_brevitas_onnx/{onnx_name}_from_brevitas.onnx'
Path(brevitas_onnx_path).parent.mkdir(parents=True, exist_ok=True)

tidy_path = f'./02_tidy_onnx/{onnx_name}_tidy.onnx'
Path(tidy_path).parent.mkdir(parents=True, exist_ok=True)

preproc_path = f'./03_preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)

postproc_path = f'./04_postproc_onnx/{onnx_name}_postproc.onnx'
Path(postproc_path).parent.mkdir(parents=True, exist_ok=True)

streamlined_path = f'./05_streamlined_onnx/{onnx_name}_streamlined.onnx'
Path(streamlined_path).parent.mkdir(parents=True, exist_ok=True)

ready_for_hw_path = f'./06_ready_for_hw_conversion_onnx/{onnx_name}_ready_for_hw_conversion.onnx'
Path(ready_for_hw_path).parent.mkdir(parents=True, exist_ok=True)

hw_layers_path = f'./07_with_hw_layers/{onnx_name}_hw.onnx'
Path(hw_layers_path).parent.mkdir(parents=True, exist_ok=True)

df_part_path = f'./08_df_part/{onnx_name}_df_part.onnx'
Path(df_part_path).parent.mkdir(parents=True, exist_ok=True)
df_part_isolated_path = f'./08_df_part/{onnx_name}_df_part_isolated.onnx'
Path(df_part_isolated_path).parent.mkdir(parents=True, exist_ok=True)

parent_model = ModelWrapper(df_part_path)
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")

pynq_board = "Pynq-Z2"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10  # 100 MHz. This is the standard clock frequency used in the FINN tutorial.

fpga_paths = {}
folded_paths = {}
for i, imp_type in enumerate(['hls', 'rtl']):
    
    fpga_path = f'./09-{i+1}_fpga_{imp_type}/{onnx_name}_fpga_{imp_type}.onnx'
    Path(fpga_path).parent.mkdir(parents=True, exist_ok=True)
    fpga_paths[imp_type] = fpga_path

    # folded_model_path = f'./10-{i+1}_folded_model_{imp_type}/{onnx_name}_folded_model_{imp_type}.onnx'
    # Path(folded_model_path).parent.mkdir(parents=True, exist_ok=True)
    # folded_paths[imp_type] = folded_model_path

# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z2"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10  # 100 MHz. This is the standard clock frequency used in the FINN tutorial.
TARGET_FPS = 30
target_clk_cycles_per_frame = 1/(TARGET_FPS * 10e-9)

/home/jose-vitor/finn/deps/brevitas/src/brevitas/__init__.py:11: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import DistributionNotFound
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('sphinxcontrib')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pack

# Convert Brevitas exported QONNX to FINN

ModelWrapper is a wrapper around the ONNX model which provides several helper functions to make it easier to work with the model.

ConvertQONNXtoFINN Convert the model to the FINN-ONNX model. The main difference of this ONNX from the standard is how the quantization is handled.

In [2]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.qonnx.infer_quant_avg_pool_2d import AvgPoolAndTruncToQuantAvgPool
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from pathlib import Path
from config import BIT_WIDTH

# Setup Path
onnx_path = f'../onnx/quant_model_{BIT_WIDTH}_bits.onnx'
onnx_name = Path(onnx_path).stem

# Note: The original tfc_end2end_example.ipynb does not run the InferShapes and InferDataTypes transformations, but without them the ConvertQONNXtoFINN transformation will fail with an error of missing shape information.
qonnx_cleanup(onnx_path, out_file=onnx_path)
model = ModelWrapper(onnx_path)
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
model = model.transform(ConvertQONNXtoFINN())

brevitas_onnx_path = f'./01_from_brevitas_onnx/{onnx_name}_from_brevitas.onnx'
Path(brevitas_onnx_path).parent.mkdir(parents=True, exist_ok=True)
model.save(brevitas_onnx_path)

In [3]:
showInNetron(brevitas_onnx_path)

Serving './01_from_brevitas_onnx/quant_model_w4a4_bits_from_brevitas.onnx' at http://0.0.0.0:8081


# Setup Model for FINN

* Tidy up (and also after EACH step)
* Pre (data feed) / Post proc (top k)
* Model streamlining (Main step) + smaller example
* Model HW Layers (Generates Matrix Vector Activation Units for fc layers)
* Model data flow partitions (Generate a sub-graph for all HW convertible nodes)
* Specialize layer, ready for hw conversion (generates hls for the dataflow partition node)

### Tidy Up

In [4]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.core.modelwrapper import ModelWrapper
from config import IM_SIZE

model = ModelWrapper(brevitas_onnx_path)

# TIDY UP
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
tidy_path = f'./02_tidy_onnx/{onnx_name}_tidy.onnx'
Path(tidy_path).parent.mkdir(parents=True, exist_ok=True)
model.save(tidy_path)

### Pre processing

FINN model expects UINT8 input. According to Xilinx, this is highly beneficial for performance, because you can directly input raw data to the model, instead of relying on CPU for pre processing.

The the QFast-SCNN model exported to QONNX has the pre processing layers integrated in the Pytorch model, so the expected input is already from 0 to 255.

If the target model was trained with tensor inputs different than [0, 255], like the standard torch.Tensor [0, 1] or tensors with Imagenet normalization, you have two main options to follow:
* Modify your Pytorch model only for the QONNX export, integrating the pre processing inside the model (the option I have chosen for QFast-SCNN).
* Add the preprocessing layers in the QONNX model, following the "Adding Pre- and Postprocessing" section of Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook.

In [5]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from qonnx.util.cleanup import cleanup as qonnx_cleanup
import torch
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from onnx import helper

# PRE PROC : NONE
model = ModelWrapper(tidy_path)

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# verify if the sizes of first and last node outputs are correct
first_node_out = model.graph.node[0].output[0]
last_node_out = model.graph.node[-1].output[0]
print(f"\nOutput shape shape of first node ({model.graph.node[0].op_type}): {model.get_tensor_shape(first_node_out)}")
print(f"Output shape shape of last node ({model.graph.node[-1].op_type}): {model.get_tensor_shape(last_node_out)}")

# verify if input and output datatypes are correct
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")
print(f"Output datatype: {model.get_tensor_datatype(last_node_out)}")

# Save the preprocessed model
preproc_path = f'./03_preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(preproc_path)

# Print a human readable representation of the graph
#print(helper.printable_graph(model.graph))


Output shape shape of first node (Mul): [1, 3, 256, 256]
Output shape shape of last node (Mul): [1, 4, 32, 32]

Input datatype: UINT8
Output datatype: FLOAT32


In [6]:
showInNetron(preproc_path)

Stopping http://0.0.0.0:8081
Stopping http://0.0.0.0:8081
Serving './03_preproc_onnx/quant_model_w4a4_bits_preproc.onnx' at http://0.0.0.0:8081


### Compare the Outputs of Pytorch Model and QONNX Model

Optional but highly recommended step. The Pytorch model will be loaded with the "finn" mode so the test input of this model is the same as the QONNX model.

In [7]:
import torch
from torchvision import transforms
from torchvision.datasets import Cityscapes
from my_finn_utils import load_state_dict, generate_cityscapes_labels, IdToTrainIdTransform
import models.QFastSCNN as qfscnn
from config import NUM_CLASSES, DATA_PATH, BIT_WIDTH, IM_HEIGHT

lable_conversion, id_names = generate_cityscapes_labels()

# Defining the Cityscapes validation dataset.
val_dataset = Cityscapes(
    root=DATA_PATH,
    split='val',
    mode='fine',
    target_type='semantic',
    transform=transforms.Compose([
        transforms.PILToTensor(), # Converting the PIL images to tensors, keeping the original pixel values (0-255) which is important for the quantized model that expects UINT8 inputs.
        transforms.CenterCrop(IM_HEIGHT) # center crop of 1024x1024
    ]),
    target_transform=transforms.Compose([
        transforms.PILToTensor(), # Converting the PIL masks to tensors, keeping the original pixel values (0-255).
        transforms.CenterCrop(IM_HEIGHT), # center crop of 1024x1024
        IdToTrainIdTransform(lable_conversion), # Converting the original Cityscapes labels to the 19 classes used for training and evaluation, as per the Cityscapes benchmark.
    ])
)

# Importing the test image and mask
img_tensor, smnt_tensor = val_dataset[0]
img_tensor = img_tensor.unsqueeze(0) # Add batch dimension
print("Input image shape:", img_tensor.shape)
print("Input image dtype:", img_tensor.dtype)
print("Input mask shape:", smnt_tensor.shape)
print("Input mask dtype:", smnt_tensor.dtype)

# Creating a Brevitas model instance and loading the quantized weights from the training phase.
brevitas_model = qfscnn.QFastSCNN(NUM_CLASSES, mode="finn")
brevitas_model = load_state_dict(brevitas_model, path=f"../train_environment/model_weights/quant_params/best_{BIT_WIDTH}_bit_quant_model.pth", strict=False)
brevitas_model.eval();

/home/jose-vitor/finn/deps/brevitas/src/brevitas/__init__.py:11: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import DistributionNotFound
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('sphinxcontrib')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pack

Input image shape: torch.Size([1, 3, 256, 256])
Input image dtype: torch.uint8
Input mask shape: torch.Size([1, 256, 256])
Input mask dtype: torch.uint8
Carregando modelo best_w4a4_bit_quant_model


In [8]:
import torch.nn.functional as F

# Run a foward pass on Brevitas model
with torch.inference_mode():
    brevitas_output = brevitas_model(img_tensor)
brevitas_output_upsampled = F.interpolate(brevitas_output, size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
brevitas_output_mask = torch.softmax(brevitas_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (brevitas_output_mask == smnt_tensor).sum().item()

print(f"Output shape from Brevitas model: {brevitas_output.shape}\n"
      f"Output shape after upsampling: {brevitas_output_upsampled.shape}\n"
      f"Output mask shape: {brevitas_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")

Output shape from Brevitas model: torch.Size([1, 4, 32, 32])
Output shape after upsampling: torch.Size([1, 4, 256, 256])
Output mask shape: torch.Size([1, 256, 256])
Accuracy: 68.93%



/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1645: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at /pytorch/c10/core/TensorImpl.h:1939.)
  return super().rename(names)


In [9]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph

model = ModelWrapper(preproc_path)
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# Run the QONNX inference.
input_tensor = img_tensor.detach().numpy().astype(np.float32) # ONNX runtime expects the input tensor to be of type float32, so we convert it to that type before passing it to the model.
input_dict = {"global_in": input_tensor}
output_dict = oxe.execute_onnx(model, input_dict)
qonnx_output = output_dict[list(output_dict.keys())[0]]

# Upsampling the QONNX output to compare to the ground truth mask from the Brevitas model.
qonnx_output_upsampled = F.interpolate(torch.from_numpy(qonnx_output), size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
qonnx_output_mask = torch.softmax(qonnx_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (qonnx_output_mask == smnt_tensor).sum().item()

print(f"Output shape from QONNX model: {qonnx_output.shape}\n"
      f"Output shape after upsampling: {qonnx_output_upsampled.shape}\n"
      f"Output mask shape: {qonnx_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")


Input datatype: UINT8


/home/jose-vitor/finn/deps/qonnx/src/qonnx/util/basic.py:316: UserWarning: The values of tensor MultiThreshold_0_out0 can't be represented with the set datatype annotation (INT4), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn/deps/qonnx/src/qonnx/util/basic.py:316: UserWarning: The values of tensor MultiThreshold_8_out0 can't be represented with the set datatype annotation (INT4), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn/deps/qonnx/src/qonnx/util/basic.py:316: UserWarning: The values of tensor MultiThreshold_11_out0 can't be represented with the set datatype annotation (INT4), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn/deps/qonnx/src/qonnx/util/basic.py:316: UserWarning: The values of tensor MultiThreshold_14_out0 can't be represented with the set datatype annotation (INT4), they will be rounded to match the datatype annotation.
  warni

Output shape from QONNX model: (1, 4, 32, 32)
Output shape after upsampling: torch.Size([1, 4, 256, 256])
Output mask shape: torch.Size([1, 256, 256])
Accuracy: 68.93%



/home/jose-vitor/finn/deps/qonnx/src/qonnx/util/basic.py:316: UserWarning: The values of tensor MultiThreshold_23_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


In [10]:
# check output types
#print(f"Brevitas output dtype: {brevitas_output.dtype}\nQONNX output dtype: {qonnx_output.dtype}\n")

# check if outputs are close enough
matching_pixels = (brevitas_output_mask == qonnx_output_mask).sum().item()
total_pixels = brevitas_output_mask.numel()
print(f"Matching pixels: {matching_pixels}/{total_pixels} ({(100 * matching_pixels/total_pixels):.2f}%)\n")

Matching pixels: 65536/65536 (100.00%)



### Post-Processing

In [11]:
# POST PROC
model = ModelWrapper(preproc_path)

# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

# Save the postprocessed model
postproc_path = f'./04_postproc_onnx/{onnx_name}_postproc.onnx'
Path(postproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(postproc_path)

### Model streamlining

Streamlining transformations listed in the bellow cell has the goal to eliminate floating point operations by moving them around and collapsing them in the previous step transforming them into multi-thresholding nodes.

In [14]:
from finn.transformation.streamline import Streamline
showSrc(Streamline)

class Streamline(Transformation):
    """Apply the streamlining transform, see arXiv:1709.04060."""

    def apply(self, model):
        streamline_transformations = [
            ConvertSubToAdd(),
            ConvertDivToMul(),
            BatchNormToAffine(),
            ConvertSignToThres(),
            MoveMulPastMaxPool(),
            AbsorbScalarBiasIntoMultiThreshold(),
            MoveScalarLinearPastInvariants(),
            MoveAddPastMul(),
            MoveScalarAddPastMatMul(),
            MoveAddPastConv(),
            MoveScalarMulPastMatMul(),
            MoveScalarMulPastConv(),
            MoveAddPastMul(),
            CollapseRepeatedAdd(),
            CollapseRepeatedMul(),
            MoveMulPastMaxPool(),
            AbsorbAddIntoMultiThreshold(),
            FactorOutMulSignMagnitude(),
            AbsorbMulIntoMultiThreshold(),
            Absorb1BitMulIntoMatMul(),
            Absorb1BitMulIntoConv(),
        ]
        for trn in streamline_transformations:
   

In [12]:
from finn.transformation.streamline import Streamline
# we can see the list of apllied transformations here : showSrc(Streamline)
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
import finn.transformation.streamline.absorb as absorb

model = ModelWrapper(postproc_path)

# STREAMLINE
model = model.transform(Streamline())

# Save the streamlined model
streamlined_path = f'./05_streamlined_onnx/{onnx_name}_streamlined.onnx'
Path(streamlined_path).parent.mkdir(parents=True, exist_ok=True)
model.save(streamlined_path)

# Print a human readable representation of the graph
#print(helper.printable_graph(model.graph))

In [13]:
showInNetron(streamlined_path)

Stopping http://0.0.0.0:8081
Stopping http://0.0.0.0:8081
Serving './05_streamlined_onnx/quant_model_w4a4_bits_streamlined.onnx' at http://0.0.0.0:8081


After the streamlining process you may notice some Mul and Add blocks included in your model graph on Netron. This blocks are terrible for hardware conversion since they will be converted to float32 operations.

Those blocks must be absorved in MultiThreshold blocks so that they be handled by quantization scale factor and zero point. The FINN streamline stardard transformations, which can be checked one of the previous notebook cells, do not include every use case of all ML models, so the user must look up the [FINN standard model transformations](https://finn.readthedocs.io/en/latest/source_code/finn.transformation.streamline.html#module-finn.transformation.streamline) and use the ones suitable for their model.

Regarding this project, I have used several standard transforms, bus also had to create two new ones, since AveragePool and Concat are blocks not included in the FINN transforms.

In [14]:
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.reorder import MoveMulPastFork, MoveScalarMulPastConv, MoveScalarMulPastConvTranspose, MoveMulPastDWConv, MoveAffinePastJoinConcat, MoveMulPastJoinAdd, MoveAddPastJoinAdd, MoveScalarLinearPastInvariants, MoveAddPastFork
from finn.transformation.streamline.absorb import AbsorbMulIntoMultiThreshold, AbsorbAddIntoMultiThreshold
from finn.transformation.streamline.collapse_repeated import CollapseRepeatedAdd, CollapseRepeatedMul, CollapseRepeatedOp

import custom_finn_transformations as cft

model = ModelWrapper(streamlined_path)

# running other transformations to clean up the graph as much as possible, to make it ready for the hardware conversion
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model = model.transform(MoveMulPastFork())
model = model.transform(MoveScalarMulPastConvTranspose())
model = model.transform(MoveScalarMulPastConv())
model = model.transform(MoveMulPastDWConv())
model = model.transform(MoveAddPastJoinAdd())
model = model.transform(MoveMulPastJoinAdd())
model = model.transform(MoveAddPastFork())
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(MoveAffinePastJoinConcat()) # For the first concat layer
model = model.transform(CollapseRepeatedAdd())
model = model.transform(CollapseRepeatedMul())
model = model.transform(AbsorbMulIntoMultiThreshold())
model = model.transform(AbsorbAddIntoMultiThreshold())

# Cleaning up the graph and inferring shapes again, to make it ready for the hardware conversion.
model = model.transform(RemoveUnusedTensors())
model = model.transform(RoundAndClipThresholds())

# Save the streamlined model
ready_for_hw_path = f'./06_ready_for_hw_conversion_onnx/{onnx_name}_ready_for_hw_conversion.onnx'
Path(ready_for_hw_path).parent.mkdir(parents=True, exist_ok=True)
model.save(ready_for_hw_path)

# Print a human readable representation of the graph
#print(helper.printable_graph(model.graph))

/home/jose-vitor/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:151: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [15]:
showInNetron(ready_for_hw_path)

Stopping http://0.0.0.0:8081
Stopping http://0.0.0.0:8081
Serving './06_ready_for_hw_conversion_onnx/quant_model_w4a4_bits_ready_for_hw_conversion.onnx' at http://0.0.0.0:8081


### Compare the Outputs of Ready for HW Convertion ONNX Model with Outputs of Pre-Proc QONNX Model

Another optional step but highly recommended if the model has unusual layers to FINN framework or if custom QONNX transforms were added to the streamline process (which is the case for my implementation of QFast-SCNN)

In [16]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

model = ModelWrapper(ready_for_hw_path)
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")

# Run the QONNX inference.
input_tensor = img_tensor.detach().numpy().astype(np.float32) # ONNX runtime expects the input tensor to be of type float32, so we convert it to that type before passing it to the model.
input_dict = {"global_in": input_tensor}
output_dict = oxe.execute_onnx(model, input_dict)
hw_qonnx_output = output_dict[list(output_dict.keys())[0]]

# Upsampling the QONNX output to compare to the ground truth mask from the Brevitas model.
hw_qonnx_output_upsampled = F.interpolate(torch.from_numpy(hw_qonnx_output), size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
hw_qonnx_output_mask = torch.softmax(hw_qonnx_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (hw_qonnx_output_mask == smnt_tensor).sum().item()

print(f"Output shape from QONNX model: {hw_qonnx_output.shape}\n"
      f"Output shape after upsampling: {hw_qonnx_output_upsampled.shape}\n"
      f"Output mask shape: {hw_qonnx_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")


Input datatype: UINT8
Output shape from QONNX model: (1, 4, 32, 32)
Output shape after upsampling: torch.Size([1, 4, 256, 256])
Output mask shape: torch.Size([1, 256, 256])
Accuracy: 68.93%



In [18]:
# check output types
#print(f"Brevitas output dtype: {brevitas_output.dtype}\nQONNX output dtype: {qonnx_output.dtype}\n")

# check if outputs are close enough
matching_pixels = (hw_qonnx_output_mask == brevitas_output_mask).sum().item()
total_pixels = hw_qonnx_output_mask.numel()
print(f"Comparando modelo Brevitas com modelo QONNX com camadas de hardware adicionadas:\n"
    f"Matching pixels: {matching_pixels}/{total_pixels} ({(100 * matching_pixels/total_pixels):.2f}%)\n")

# check if outputs are close enough
matching_pixels = (hw_qonnx_output_mask == qonnx_output_mask).sum().item()
total_pixels = hw_qonnx_output_mask.numel()
print(f"Comparando modelo QONNX com camadas de hardware adicionadas com modelo QONNX sem camadas de hardware:\n"
    f"Matching pixels: {matching_pixels}/{total_pixels} ({(100 * matching_pixels/total_pixels):.2f}%)\n")

Comparando modelo Brevitas com modelo QONNX com camadas de hardware adicionadas:
Matching pixels: 65536/65536 (100.00%)

Comparando modelo QONNX com camadas de hardware adicionadas com modelo QONNX sem camadas de hardware:
Matching pixels: 65536/65536 (100.00%)



## Convert to HW

This step does not generate HLS nor RTL code, but rather merges thresholds and matrix vector operations (or convolutions if you use conv nets) into Matrix Vector Activation Units (or Sliding Window Units), if possible, that each represent a layer that FINN can easily work with. Note that an input preprocessing quantiazer will be implemented as a simple standalone Threasholding layer that will then nbe converted to Thresholding_hls layer.

In [19]:
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.absorb import AbsorbConsecutiveTransposes, AbsorbTransposeIntoMultiThreshold, AbsorbTransposeIntoResize, AbsorbConsecutiveTransposes
from finn.transformation.streamline.reorder import MoveTransposePastJoinConcat, MoveTransposePastJoinAdd
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
import custom_finn_transformations as cft

# TO HW LAYERS

model = ModelWrapper(ready_for_hw_path)
model = model.transform(LowerConvsToMatMul())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(to_hw.InferChannelwiseLinearLayer())
model = model.transform(to_hw.InferPool())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferVectorVectorActivation())
model = model.transform(AbsorbTransposeIntoMultiThreshold())
model = model.transform(to_hw.InferThresholdingLayer())

# Adjustments to the model to make it compatible with the hardware conversion, including reordering and absorbing operations.
model = model.transform(InferShapes())
model = model.transform(InferDataLayouts())
model = model.transform(AbsorbTransposeIntoResize())
model = model.transform(to_hw.InferUpsample())
model = model.transform(AbsorbTransposeIntoResize())
model = model.transform(AbsorbTransposeIntoMultiThreshold())
model = model.transform(MoveTransposePastJoinConcat())
model = model.transform(AbsorbConsecutiveTransposes())

# Runs the cuustom transformation to transform Concat nodes from NCHW to NHWC, and them convert them to hardware compatible layers.
model = model.transform(cft.MakeElementwiseAddNHWC())
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
model = model.transform(to_hw.InferConcatLayer())

# Insert InferDuplicateStreamsLayer to deal with skip connections in the model, which are not supported by the hardware conversion.
model = model.transform(to_hw.InferDuplicateStreamsLayer())
model = model.transform(RemoveUnusedTensors())

# Replace last two ElementwiseAdd and ElementwiseMul nodes with Add and Mul nodes, to be able to create teh dataflow partition.
model = model.transform(cft.RevertNCHWElementwiseToONNX())

# Save the hw model
hw_layers_path = f'./07_with_hw_layers/{onnx_name}_hw.onnx'
Path(hw_layers_path).parent.mkdir(parents=True, exist_ok=True)
model.save(hw_layers_path)

#print(helper.printable_graph(model.graph))

/home/jose-vitor/finn/deps/qonnx/src/qonnx/core/modelwrapper.py:244: DeprecationWarning: InferChannelwiseLinearLayer is deprecated. Use InferElementwiseBinaryOperation instead. ChannelwiseOp is being replaced by ElementwiseBinary operations.
  (transformed_model, model_was_changed) = transformation.apply(
/home/jose-vitor/finn/src/finn/custom_op/fpgadataflow/fmpadding.py:123: UserWarning: inputDataType changing for FMPadding_Batch_: UINT4 -> FLOAT32 
  warnings.warn(warn_str)
/home/jose-vitor/finn/src/finn/custom_op/fpgadataflow/convolutioninputgenerator.py:140: UserWarning: ConvolutionInputGenerator_: inputDataType changing from UINT4 to FLOAT32
  warnings.warn(
/home/jose-vitor/finn/src/finn/custom_op/fpgadataflow/convolutioninputgenerator.py:150: UserWarning: ConvolutionInputGenerator_: outputDataType changing from UINT4 to FLOAT32
  warnings.warn(
/home/jose-vitor/finn/src/finn/custom_op/fpgadataflow/vectorvectoractivation.py:176: UserWarning: inputDataType changing for VVAU_: UINT

In [20]:
showInNetron(hw_layers_path)

Stopping http://0.0.0.0:8081
Stopping http://0.0.0.0:8081
Serving './07_with_hw_layers/quant_model_w4a4_bits_hw.onnx' at http://0.0.0.0:8081


### Isolate HW convertible layers.

Once MVAU are infered, we can ask FINN to discriminate HW convertible layers into dataflow partitions.

In [21]:
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition

model = ModelWrapper(hw_layers_path)
parent_model = model.transform(CreateDataflowPartition())

# Save the dataflow partition model
df_part_path = f'./08_df_part/{onnx_name}_df_part.onnx'
Path(df_part_path).parent.mkdir(parents=True, exist_ok=True)
parent_model.save(df_part_path)

In [22]:
showInNetron(df_part_path)

Stopping http://0.0.0.0:8081
Stopping http://0.0.0.0:8081
Serving './08_df_part/quant_model_w4a4_bits_df_part.onnx' at http://0.0.0.0:8081


We select the child streaming dataflow model to work on it (i.e. do the actual HLS/RTL conversion)

In [23]:
from qonnx.custom_op.registry import getCustomOp
parent_model = ModelWrapper(df_part_path)
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")

### MVAU HLS Conversion

With the entire model isolated with only HW abstraction layers we can replace them by actual HLS or RTL module using FINN. This is done by calling the Specializelayers Transform. It is also possible before that to let the FINN flow now if the preference of implementation is HLS layers or RTL layers by setting the nodeattr "preferred_impl_style" as "hls" or "rtl". If this attr is left empty, than by default the preferred implementation will be HLS. If RTL and FINN does not have the implementation of this node on RTL, then it will be set back to HLS.

For this project I will do both options to see the diferences.

In [24]:
# print the names of the supported PYNQ boards
from finn.util.basic import pynq_part_map
print(pynq_part_map.keys())

# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z2"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10  # 100 MHz. This is the standard clock frequency used in the FINN tutorial.

dict_keys(['Ultra96', 'Ultra96-V2', 'Pynq-Z1', 'Pynq-Z2', 'ZCU102', 'ZCU104', 'ZCU111', 'RFSoC2x2', 'RFSoC4x2', 'KV260_SOM', 'AUP-ZU3_8GB'])


In [25]:
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds

fpga_paths = {}
for i, imp_type in enumerate(['hls', 'rtl']):
    print(f"Generating FPGA model for implementation style: {imp_type}")
    model = ModelWrapper(dataflow_model_filename)

    # Set all preferred_impl_style attributes to "imp_stype".
    for node in model.graph.node:
        node_inst = getCustomOp(node)
        node_inst.set_nodeattr("preferred_impl_style", imp_type)

    model = model.transform(SpecializeLayers(fpga_part))

    # Save the FPGA model
    fpga_path = f'./09-{i+1}_fpga_{imp_type}/{onnx_name}_fpga_{imp_type}.onnx'
    Path(fpga_path).parent.mkdir(parents=True, exist_ok=True)
    model.save(fpga_path)
    fpga_paths[imp_type] = fpga_path

Generating FPGA model for implementation style: hls
Generating FPGA model for implementation style: rtl


/home/jose-vitor/finn/src/finn/transformation/fpgadataflow/specialize_layers.py:124: UserWarning: There is no HLS variant of ConvolutionInputGenerator. Node ConvolutionInputGenerator_ will automatically be
                        set to RTL variant.
  warnings.warn(warn_str)
/home/jose-vitor/finn/src/finn/transformation/fpgadataflow/specialize_layers.py:124: UserWarning: There is no HLS variant of FMPadding. Node FMPadding_Batch_ will automatically be
                        set to RTL variant.
  warnings.warn(warn_str)
/home/jose-vitor/finn/src/finn/transformation/fpgadataflow/specialize_layers.py:168: UserWarning: There is no RTL variant for VVAU_. The node will automatically be
                        set to HLS variant. Please check the bit-widths to be <= 8 and ensure the
                        thresholds are implemented as standalone layer. Note that the RTL-variant
                        of this layer is only supported on Versal boards
  warnings.warn(warn_str)
/home/jose-vito

In [41]:
showInNetron(fpga_paths['hls'])

Stopping http://0.0.0.0:8081
Stopping http://0.0.0.0:8081
Serving './09-1_fpga_hls/quant_model_w4a4_bits_fpga_hls.onnx' at http://0.0.0.0:8081


In [40]:
showInNetron(fpga_paths['rtl'])

Stopping http://0.0.0.0:8081
Stopping http://0.0.0.0:8081
Serving './09-2_fpga_rtl/quant_model_w4a4_bits_fpga_rtl.onnx' at http://0.0.0.0:8081


### RUN HARDWARE IMPLEMENTATION ESTIMATES

Before entering the model folding process, we can run simulation the hardware performance and utilization.

In [28]:
import numpy as np
from qonnx.custom_op.base import CustomOp

# --- BEGIN DATA TYPE MONKEY PATCH ---
# Save the original set_nodeattr method to call it later
original_set_nodeattr = CustomOp.set_nodeattr

# Create a patched version of the set_nodeattr method that converts numpy integer and floating types to native Python int and float types
def patched_set_nodeattr(self, name, value):
    if isinstance(value, np.integer):
        value = int(value)
    elif isinstance(value, np.floating):
        value = float(value)
    return original_set_nodeattr(self, name, value)

# Inject the patched method into the CustomOp class
CustomOp.set_nodeattr = patched_set_nodeattr
# --- END OF MONKEY PATCH DE TIPAGEM ---


# ESTIMATE
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import shutil

IMP_TYPE = "both"  # "hls", "rtl" or "both"
TARGET_FPS = 30

estimates_output_dirs = {}
imp_type_array = ["hls", "rtl"] if IMP_TYPE == "both" else [IMP_TYPE]
for imp_type in imp_type_array:
    estimates_output_dir = f"./finn-estimates/{imp_type}_output_estimates_only"
    estimates_output_dirs[imp_type] = estimates_output_dir

    #Delete previous run results if exist
    if os.path.exists(estimates_output_dir):
        shutil.rmtree(estimates_output_dir)
        print("Previous run results deleted!")

    cfg_estimates = build.DataflowBuildConfig(
        output_dir          = estimates_output_dir,
        mvau_wwidth_max     = 256,
        target_fps          = TARGET_FPS,
        synth_clk_period_ns = 10.0,
        fpga_part           = fpga_part,
        steps               = build_cfg.estimate_only_dataflow_steps,
        generate_outputs=[
            build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        ]
    )

    build.build_dataflow_cfg(fpga_paths[imp_type], cfg_estimates)

Building dataflow accelerator from ./09-1_fpga_hls/quant_model_w4a4_bits_fpga_hls.onnx
Intermediate outputs will be generated in /tmp/finn_dev_jose-vitor
Final outputs will be generated in ./finn-estimates/hls_output_estimates_only
Build log is at ./finn-estimates/hls_output_estimates_only/build_dataflow.log
FINN Build Configuration Check Report
Timestamp: 2026-10-31T23:01:25.628488
Vivado: 2022.2
----------------------------------------------------------------------

WARNINGS:
  [!] standalone_thresholds: standalone_thresholds=False: MatMul+MultiThreshold patterns will be fused into MVAU with output activation, which cannot use RTL MVAU. In those cases HLS MVAU will be used instead
      -> Set standalone_thresholds=True if you want to make sure to always get the RTL MVAU implementation

SUMMARY: 0 errors, 1 warnings, 0 info
Configuration check report saved to: ./finn-estimates/hls_output_estimates_only/config_check_report.txt
Running step: phase_prepare_model [1/4]
Running step: phas

### Read the estimates reports

In these reports, we can see the estimates fits our target, and we can get an overview of hw many LUTs will be used

In [29]:
IMP_TYPE = "both"  # "hls", "rtl" or "both"
imp_type_array = ["hls", "rtl"] if IMP_TYPE == "both" else [IMP_TYPE]

estimates_output_dirs = {}
for imp_type in ["hls", "rtl"]:
    estimates_output_dir = f"./finn-estimates/{imp_type}_output_estimates_only"
    estimates_output_dirs[imp_type] = estimates_output_dir

In [30]:
for imp_type in imp_type_array:
    print(f"\n[!] ESTIMATIVAS DE PERFORMANCE PARA IMPLEMENTAÇÃO {imp_type}:")
    ! cat {estimates_output_dirs["hls"]}/report/estimate_network_performance.json


[!] ESTIMATIVAS DE PERFORMANCE PARA IMPLEMENTAÇÃO hls:
{
  "critical_path_cycles": 44524788,
  "max_cycles": 3145728,
  "max_cycles_node_name": "MVAU_hls_1",
  "estimated_throughput_fps": 31.789143880208332,
  "estimated_latency_ns": 445247880.0
}
[!] ESTIMATIVAS DE PERFORMANCE PARA IMPLEMENTAÇÃO rtl:
{
  "critical_path_cycles": 44524788,
  "max_cycles": 3145728,
  "max_cycles_node_name": "MVAU_hls_1",
  "estimated_throughput_fps": 31.789143880208332,
  "estimated_latency_ns": 445247880.0
}

In [31]:
from my_finn_utils import read_json_dict, pynqz2_viability_check

for imp_type in imp_type_array:
    resource_dict = read_json_dict(f"{estimates_output_dirs[imp_type]}/report/estimate_layer_resources.json")
    print(f"\n[!] VERIFICANDO VIABILIDADE DE RECURSOS PARA O TIPO DE IMPLEMENTAÇÃO: {imp_type}")
    pynqz2_viability_check(resource_dict)


[!] VERIFICANDO VIABILIDADE DE RECURSOS PARA O TIPO DE IMPLEMENTAÇÃO: hls
Pynq-Z2 uses ZYNQ XC7Z020-1CLG400C SoC
Resource usage totals: {'BRAM_18K': 96.0, 'LUT': 31774.0, 'URAM': 0.0, 'DSP': 0.0}

Checking LUT: 31774.0 used, 53200.0 limit. Usage percentage: 59.73%. PASS
Checking BRAM_18K: 96.0 used, 280.0 limit. Usage percentage: 34.29%. PASS
Checking DSP: 0.0 used, 220.0 limit. Usage percentage: 0.00%. PASS
Checking URAM: 0.0 used, 0.0 limit. Usage percentage: 0.00%. PASS


[!] VERIFICANDO VIABILIDADE DE RECURSOS PARA O TIPO DE IMPLEMENTAÇÃO: rtl
Pynq-Z2 uses ZYNQ XC7Z020-1CLG400C SoC
Resource usage totals: {'BRAM_18K': 96.0, 'LUT': 20116.0, 'URAM': 0.0, 'DSP': 50.0}

Checking LUT: 20116.0 used, 53200.0 limit. Usage percentage: 37.81%. PASS
Checking BRAM_18K: 96.0 used, 280.0 limit. Usage percentage: 34.29%. PASS
Checking DSP: 50.0 used, 220.0 limit. Usage percentage: 22.73%. PASS
Checking URAM: 0.0 used, 0.0 limit. Usage percentage: 0.00%. PASS



In [ ]:
'''HLS'''
#! cat {estimates_output_dirs["hls"]}/report/estimate_layer_resources.json
#! cat {estimates_output_dirs["hls"]}/report/estimate_layer_cycles.json

'''RTL'''
! cat {estimates_output_dirs["rtl"]}/report/estimate_layer_resources.json
#! cat {estimates_output_dirs["rtl"]}/report/estimate_layer_cycles.json

### FOLDING MODEL

Folding in FINN describes how much a layer is time-multiplexed in terms of execution resources. There are several folding factors for each layer, controlled by the PE (parallelization over outputs) and SIMD (parallelization over inputs) parameters as described by the original [FINN paper](https://arxiv.org/pdf/1612.07119). The higher the PE and SIMD values are set, the faster the generated accelerator will run, and the more FPGA resources it will consume.

- PE = Processing Elements => aka actual computing units that will run MAC operations.
    - Theorically improves computing throughput.

- SIMD = Single Instruction Multiple Data => aka a way to feed MULTIPLE data at ONCE in a signle PE
    - Theorically improves datapath bandwidth.

One can set folding parameters manually by following the steps of the FINN example notebook [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb). 

This project is using the SetFolding transform to define automatically the PE ans SIMD parameter depending on target requirements. This is being done because it worked very well for the optimized Fast-SCNN model.

If your project requires more tinkering regarding folding, then I will leave the research for you.

In [32]:
'''
THIS CELL ONLY CHECKS IF THE HARDWARE IMPLEMENTATION OF THE MODEL IS FEASIBLE, BY CHECKING IF THE THRESHOLDS OF THE MULTITHRESHOLD NODES ARE WITHIN THE RANGE OF THE HARDWARE DATATYPE. IT DOES NOT MODIFY THE MODEL IN ANY WAY.
'''

from finn.transformation.fpgadataflow.set_folding import SetFolding
from finn.transformation.fpgadataflow.set_fifo_depths import InsertAndSetFIFODepths
from qonnx.custom_op.registry import getCustomOp

IMP_TYPE = "both"  # "hls", "rtl" or "both"
INDEX_SUM = 2 if IMP_TYPE == "rtl" else 1

TARGET_FPS = 30
target_clk_cycles_per_frame = 1/(TARGET_FPS * 10e-9)

imp_type_array = ["hls", "rtl"] if IMP_TYPE == "both" else [IMP_TYPE]
for i, imp_type in enumerate(imp_type_array):
    fpga_path = fpga_paths[imp_type]
    model = ModelWrapper(fpga_path)
    # Actual method SetFolding that will automatically modify the folding settings

    print(f"--- SEARCHING FOR HARDWARE BOTTLENECKS FOR IMPLEMENTATION: {imp_type} ---")
    exception_raised = False
    for node in model.graph.node:
        if node.op_type == f"Thresholding_{imp_type}":
            inst = getCustomOp(node)
            try:
                if imp_type == "hls":
                    inst.code_generation_ipgen(model, fpga_part, target_clk_cycles_per_frame)
                elif imp_type == "rtl":
                    inst.prepare_codegen_rtl_values(model)
            except Exception as e:
                if type(e) == AssertionError:
                    exception_raised = True
                    print(f"\n[!] FAILED ON NODE: {node.input[1]}: {e}")
                    wdt = inst.get_nodeattr("weightDataType")
                    print(f" -> Required Hardware Dtype: {wdt}.")
                    param_name = node.input[1]
                    thresh = model.get_initializer(param_name)
                    print(f" -> Values present in the threshold: MIN: {thresh.min()}, MAX: {thresh.max()}. Weights OK: {DataType[wdt].allowed(thresh.min()) and DataType[wdt].allowed(thresh.max())}.")
                    print(f" -> First 10 thresholds: {thresh.flatten()[:10]}\n"
                        f" -> Dtype of the threshold: {thresh.dtype}\n")
    if not exception_raised:
        print("ALL THRESHOLDS ARE WITHIN THE RANGE OF THE HARDWARE DATATYPE.\n")

--- SEARCHING FOR HARDWARE BOTTLENECKS FOR IMPLEMENTATION: hls ---
ALL THRESHOLDS ARE WITHIN THE RANGE OF THE HARDWARE DATATYPE.

--- SEARCHING FOR HARDWARE BOTTLENECKS FOR IMPLEMENTATION: rtl ---
ALL THRESHOLDS ARE WITHIN THE RANGE OF THE HARDWARE DATATYPE.



After the SetFolding transform

In [3]:
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.fpgadataflow.set_folding import SetFolding
from finn.transformation.fpgadataflow.set_fifo_depths import InsertAndSetFIFODepths
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from qonnx.custom_op.registry import getCustomOp
from qonnx.custom_op.base import CustomOp

import custom_finn_transformations as cft

IMP_TYPE = "rtl"  # "hls", "rtl", or "both"
INSERT_FIFO_METHOD = "rtlsim"  # "fixed" or "rtlsim"

TARGET_FPS = 30
target_clk_cycles_per_frame = 1/(TARGET_FPS * 10e-9)
INDEX_SUM = 2 if IMP_TYPE == "rtl" else 1

folded_paths = {}
assert IMP_TYPE in ["hls", "rtl", "both"], "IMP_TYPE must be 'hls', 'rtl' or 'both'."
imp_type_array = ["hls", "rtl"] if IMP_TYPE == "both" else [IMP_TYPE]
for i, imp_type in enumerate(imp_type_array):

    print(f"Generating folded model for implementation style: {imp_type}")
    model = ModelWrapper(fpga_paths[imp_type])

    # Actual method SetFolding that will automatically modify the folding settings
    # FINN may generate NumPy integer attributes (for example, SIMD=np.int64).
    # ONNX/FINN node attributes require native Python integers.
    if not getattr(CustomOp.set_nodeattr, "_native_types_patch", False):
        _original_set_nodeattr = CustomOp.set_nodeattr
        def _set_nodeattr_native_types(self, name, value):
            if isinstance(value, np.integer):
                value = int(value)
            elif isinstance(value, np.floating):
                value = float(value)
            return _original_set_nodeattr(self, name, value)
        _set_nodeattr_native_types._native_types_patch = True
        CustomOp.set_nodeattr = _set_nodeattr_native_types

    model = model.transform(
        SetFolding(
            target_cycles_per_frame=float(target_clk_cycles_per_frame),
            mvau_wwidth_max=256,
            two_pass_relaxation=True,
        )
    )

    if INSERT_FIFO_METHOD == "fixed":
        model = model.transform(
            cft.InsertFIFODepths(
                fpga_part,
                fixed_depth=34,
                ram_style="block"
            )
        )

    elif INSERT_FIFO_METHOD == "rtlsim":
        # Insert appropriate-depth StreamingFIFOs through RTLSim that preserve throughput in the created accelerator
        model = model.transform(
            InsertAndSetFIFODepths(
                fpga_part,
                clk_ns=10.0,
                max_depth=None,
                swg_exception=True, # Set True to reduce depth of FIFOs when possible, to reduce resource usage. Mandatory since max_depth is set to a high value.
                fifosim_input_throttle=True,
                debug_log_dir=f"./finn-estimates/{imp_type}_InsertAndSetFIFODepths_log"
            )
        )
    
    # Save the folded model
    folded_model_path = f'./10-{i+INDEX_SUM}_folded_model_{INSERT_FIFO_METHOD}_FIFO_{imp_type}/{onnx_name}_folded_model_{INSERT_FIFO_METHOD}_FIFO_{imp_type}.onnx'
    Path(folded_model_path).parent.mkdir(parents=True, exist_ok=True)
    model.save(folded_model_path)
    folded_paths[imp_type] = folded_model_path

Generating folded model for implementation style: rtl


/home/jose-vitor/finn/src/finn/transformation/fpgadataflow/set_folding.py:276: UserWarning: SetFolding doesn't know how to handle op_type UpsampleNearestNeighbour_hls
  warnings.warn("SetFolding doesn't know how to handle op_type " + op_type)


Vivado Simulator v2022.2
Copyright 1986-1999, 2001-2022 Xilinx, Inc. All Rights Reserved.
Running: /home/jose-vitor/Xilinx/Vivado/2022.2/bin/unwrapped/lnx64.o/xelab work.glbl work.finn_design_wrapper -relax -prj rtlsim.prj -dll -s finn_design_wrapper --mt 4 -define FINN_SIMULATION -L smartconnect_v1_0 -L axi_protocol_checker_v1_1_12 -L axi_protocol_checker_v1_1_13 -L axis_protocol_checker_v1_1_11 -L axis_protocol_checker_v1_1_12 -L xil_defaultlib -L unisims_ver -L xpm -L floating_point_v7_1_16 -L floating_point_v7_0_21 -L floating_point_v7_1_18 -L floating_point_v7_1_15 -L floating_point_v7_1_19 -L floating_point_v7_1_21 -L floating_point_v7_0_26 
Using 4 slave threads.
INFO: [VRFC 10-2263] Analyzing Verilog file "/home/jose-vitor/Xilinx/Vivado/2022.2/data/verilog/src/glbl.v" into library work
INFO: [VRFC 10-311] analyzing module glbl
INFO: [VRFC 10-2263] Analyzing SystemVerilog file "/tmp/finn_dev_jose-vitor/code_gen_ipgen_ConvolutionInputGenerator_rtl_0_i48qgo4m/swg_pkg.sv" into libr

In [ ]:
IMP_TYPE = "rtl"  # "hls", "rtl", or "both"
INSERT_FIFO_METHOD = "rtlsim"  # "fixed" or "rtlsim"

INDEX_SUM = 2 if IMP_TYPE == "rtl" else 1

folded_paths = {}
assert IMP_TYPE in ["hls", "rtl", "both"], "IMP_TYPE must be 'hls', 'rtl' or 'both'."
imp_type_array = ["hls", "rtl"] if IMP_TYPE == "both" else [IMP_TYPE]
for i, imp_type in enumerate(imp_type_array):
    # Save the folded model names for visualization in Netron.
    folded_model_path = f'./10-{i+INDEX_SUM}_folded_model_{INSERT_FIFO_METHOD}_FIFO_{imp_type}/{onnx_name}_folded_model_{INSERT_FIFO_METHOD}_FIFO_{imp_type}.onnx'
    folded_paths[imp_type] = folded_model_path

In [42]:
showInNetron(folded_paths['hls'])

Stopping http://0.0.0.0:8081
Stopping http://0.0.0.0:8081
Serving './10-1_folded_model_fixed_FIFO_hls/quant_model_w4a4_bits_folded_model_fixed_FIFO_hls.onnx' at http://0.0.0.0:8081


In [4]:
showInNetron(folded_paths['rtl'])

Serving './10-2_folded_model_rtlsim_FIFO_rtl/quant_model_w4a4_bits_folded_model_rtlsim_FIFO_rtl.onnx' at http://0.0.0.0:8081


In [7]:
model = ModelWrapper(folded_paths['rtl'])
print(helper.printable_graph(model.graph))

graph main_graph (
  %global_in[FLOAT, 1x256x256x3]
) initializers (
  %Thresholding_rtl_0_param0[FLOAT, 3x15]
  %Thresholding_rtl_1_param0[FLOAT, 32x15]
  %Thresholding_rtl_2_param0[FLOAT, 32x15]
  %Thresholding_rtl_3_param0[FLOAT, 48x15]
  %Thresholding_rtl_4_param0[FLOAT, 48x15]
  %Thresholding_rtl_5_param0[FLOAT, 64x15]
  %Thresholding_rtl_6_param0[FLOAT, 128x15]
  %Thresholding_rtl_7_param0[FLOAT, 128x15]
  %Thresholding_rtl_8_param0[FLOAT, 64x15]
  %Thresholding_rtl_9_param0[FLOAT, 128x15]
  %Thresholding_rtl_10_param0[FLOAT, 128x15]
  %Thresholding_rtl_11_param0[FLOAT, 96x15]
  %Thresholding_rtl_12_param0[FLOAT, 192x15]
  %Thresholding_rtl_13_param0[FLOAT, 192x15]
  %Thresholding_rtl_14_param0[FLOAT, 128x15]
  %Thresholding_rtl_15_param0[FLOAT, 128x15]
  %Thresholding_rtl_16_param0[FLOAT, 128x15]
  %Thresholding_rtl_17_param0[FLOAT, 128x15]
  %Thresholding_rtl_18_param0[FLOAT, 128x15]
  %Thresholding_rtl_19_param0[FLOAT, 128x15]
  %Thresholding_rtl_20_param0[FLOAT, 128x15]
  %Th

In [5]:
from qonnx.core.modelwrapper import ModelWrapper
from finn.analysis.fpgadataflow.res_estimation import res_estimation
from finn.analysis.fpgadataflow.dataflow_performance import dataflow_performance
from my_finn_utils import calculate_total_resource_usage, pynqz2_viability_check
import custom_finn_transformations as cft
import pprint

IMP_TYPE = "rtl"  # "hls", "rtl", or "both"

imp_types = ["hls", "rtl"] if IMP_TYPE == "both" else [IMP_TYPE]
for imp_type in imp_types:
    # Load the folded model for the current implementation type
    model = ModelWrapper(folded_paths[imp_type])
    print(f"\n\n--- ({imp_type}) HARDWARE RESOURCES ANALYSIS ---")
    res = res_estimation(model, fpga_part)
    #pprint.pprint(res)
    total_res = calculate_total_resource_usage(res)
    #pprint.pprint(total_res)
    pynqz2_viability_check(total_res)
    print(f"--- ({imp_type}) PERFORMANCE ESTIMATES ---")
    perf = dataflow_performance(model)
    pprint.pprint(perf)



--- (rtl) HARDWARE RESOURCES ANALYSIS ---
Pynq-Z2 uses ZYNQ XC7Z020-1CLG400C SoC
Resource usage totals: {'BRAM_18K': 111.0, 'DSP': 50.0, 'LUT': 37437.0, 'URAM': 61.0}

Checking LUT: 37437.0 used, 53200.0 limit. Usage percentage: 70.37%. PASS
Checking BRAM_18K: 111.0 used, 280.0 limit. Usage percentage: 39.64%. PASS
Checking DSP: 50.0 used, 220.0 limit. Usage percentage: 22.73%. PASS
Checking URAM: 61.0 used, 0.0 limit. Usage percentage: 0.00%. IMPOSSIBLE (resource not available on Pynq-Z2)

--- (rtl) PERFORMANCE ESTIMATES ---
{'critical_path_cycles': 44524788,
 'max_cycles': 3145728,
 'max_cycles_node_name': 'MVAU_rtl_1'}


In [6]:
res

{'StreamingFIFO_rtl_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 16,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'Thresholding_rtl_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 36,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'StreamingFIFO_rtl_1': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 156,
  'URAM': 32,
  'URAM_efficiency': 0.0912322998046875,
  'DSP': 0},
 'ConvolutionInputGenerator_rtl_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 544,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'StreamingFIFO_rtl_2': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 16,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'StreamingDataWidthConverter_rtl_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 39,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'StreamingFIFO_rtl_3': {'BRAM_18K': 5,
  'BRAM_efficiency': 15.094184027777779,
  'LUT': 177,
  'URAM': 8,
  'URAM_efficiency': 0.5896165635850694,
  'DSP': 0},
 'MVAU_rtl_0': {'BRAM_18K': 1,


## HARDWARE BUILD

### Run the actual harware build


This step can take a lot of time.

Make sure you did your dev enrionement setup right, aka all the ENV varibles that should point on the xilinx tools are sourced

In [7]:
folded_paths['rtl']

'./10-2_folded_model_rtl/quant_model_4_bits_folded_model_rtl.onnx'

In [ ]:
# Actual hardware build, ZYNQ BUILD
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from pathlib import Path

IMP_TYPE = "rtl"  # "hls", "rtl" or "both"
INDEX_SUM = 2 if IMP_TYPE == "rtl" else 1

post_synth_paths = {}
imp_type_array = ["hls", "rtl"] if IMP_TYPE == "both" else [IMP_TYPE]
for i, imp_type in enumerate(imp_type_array):

    model = ModelWrapper(folded_paths[imp_type])
    model = model.transform(
        ZynqBuild(
            platform=pynq_board,
            period_ns=target_clk_ns,
            partition_model_dir="./test",
            enable_debug=True
        )
    )

    # Save the FPGA post synthesis model
    post_synth_path = f'./11-{i+INDEX_SUM}_post_synth_{imp_type}/{onnx_name}_post_synth_{imp_type}.onnx'
    Path(post_synth_path).parent.mkdir(parents=True, exist_ok=True)
    model.save(post_synth_path)
    post_synth_paths[imp_type] = post_synth_path

/home/jose-vitor/finn/src/finn/transformation/fpgadataflow/floorplan.py:107: UserWarning: 166 nodes have no entry in the provided floorplan, SLR was set to -1
  warnings.warn(
/home/jose-vitor/finn/src/finn/transformation/fpgadataflow/insert_fifo.py:228: UserWarning: Input FIFO for IODMA_hls_0_out0 has depth 2 and won't
                        be created. This may cause RTL simulation issues.
                        
  warnings.warn(
/home/jose-vitor/finn/src/finn/transformation/fpgadataflow/insert_fifo.py:287: UserWarning: Output FIFO for MVAU_rtl_24_out0 has depth 2 and won't
                        be created. This may cause RTL simulation issues.
                        
  warnings.warn(
/home/jose-vitor/finn/src/finn/transformation/fpgadataflow/create_stitched_ip.py:417: UserWarning: First node is not StreamingFIFO or IODMA.
                You may experience incorrect stitched-IP rtlsim or hardware
                behavior. It is strongly recommended to insert FIFOs prior to
    

KeyboardInterrupt: 

In [7]:
showInNetron(post_synth_path)

NameError: name 'post_synth_path' is not defined